# YOLO_DRT API — бенч (фазы + память процесса)

Каждый прогон: **тайминги фаз** + **память именно процесса API** (не «вся карта» из NVML).

| Фаза | Что это |
|------|---------|
| upload/create | HTTP upload или path-job |
| preload | Декод в RAM |
| infer (Pass1) | YOLO + трек |
| Pass2 | OSNet tracklet link |
| finalize | JSON / packets |

| Память | Источник |
|--------|----------|
| **process_rss_peak_mb** | Working Set этого PID |
| **cuda_allocated_peak_mb** | `torch.cuda` этого процесса |
| **gpu_device_peak_mb** | NVML — вся VRAM на GPU (сравнение) |

Артефакты run: `{run_id}_process_memory_samples.json`, `stats_summary.process_memory`.

Перед прогоном: **перезапусти API** (`docker compose up -d --build` или `run_api.bat`).
Результаты: `benchmark_results/` (после Run All).


In [ ]:
from pathlib import Path

# Docker API (compose) -> 8080. Host uvicorn -> 8765
API_BASE = "http://127.0.0.1:8080"

# Path на хосте (USE_UPLOAD=False) или файл в YOLO_DOCKER/videos для Docker path-job
VIDEO_PATH = Path(r"C:\Users\Shtefan\Desktop\Новая папка (6)\video.mp4")
# VIDEO_PATH = Path("/data/videos/video.mp4")  # inside container path for curl/path sync

PROMPT = "person"
MAX_DURATION_SECONDS = None

_port = API_BASE.rstrip("/").split(":")[-1]
USE_UPLOAD = False if _port == "8765" else True

WARMUP_RUN = False
REPEAT_RUNS = 0

# path + sync = fair bench (no poll storm). upload → USE_SYNC_BENCH ignored.
USE_SYNC_BENCH = True
POLL_INTERVAL_SEC = 0.5

RESULTS_DIR = Path(__file__).resolve().parent / "benchmark_results" if '__file__' in dir() else Path(r"d:\Projects\SAM3_construction\YOLO_DRT\YOLO_DOCKER\notebooks\benchmark_results")
try:
    RESULTS_DIR = Path(__file__).resolve().parent / "benchmark_results"
except NameError:
    RESULTS_DIR = Path(r"d:\Projects\SAM3_construction\YOLO_DRT\YOLO_DOCKER\notebooks\benchmark_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("API_BASE:", API_BASE, "USE_UPLOAD:", USE_UPLOAD, "SYNC:", USE_SYNC_BENCH)
print("RESULTS_DIR:", RESULTS_DIR.resolve())


In [ ]:
import json
import time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Callable

import matplotlib.pyplot as plt
import pandas as pd
import requests
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["figure.dpi"] = 110


class ApiError(RuntimeError):
    pass


@dataclass
class Tick:
    t_wall: float
    status: str
    current: int
    total: int
    percent: float
    fps: float
    elapsed_sec: float
    eta_seconds: float
    gpu_mem_mb: float
    cuda_allocated_mb: float = 0.0
    cuda_reserved_mb: float = 0.0
    process_rss_mb: float = 0.0
    gpu_device_used_mb: float = 0.0
    gpu_util_pct: float = 0.0
    instances_peak: int = 0
    phase: str = "inference"


@dataclass
class RunResult:
    job_id: str
    wall_sec: float
    status: str
    job: dict[str, Any]
    ticks: list[Tick] = field(default_factory=list)
    files: dict[str, bytes] = field(default_factory=dict)
    error: str | None = None
    label: str = "run"
    upload_sec: float = 0.0
    process_poll_sec: float = 0.0
    use_upload: bool = True

    @property
    def result(self) -> dict:
        return self.job.get("result") or {}

    @property
    def record(self) -> dict:
        return self.result.get("record") or {}

    @property
    def stats(self) -> dict:
        return self.record.get("stats_summary") or {}

    @property
    def pipeline(self) -> dict:
        return self.record.get("pipeline") or {}


def _extract_speed(run: RunResult) -> dict[str, Any]:
    """Единая таблица метрик: video duration vs wall, YOLO vs Pass2."""
    rec = run.record
    res = run.result
    st = run.stats
    pipe = run.pipeline
    progress = (run.job.get("progress") or {}) if run.job else {}

    frames = int(rec.get("frames") or res.get("frames") or 0)
    src = int(
        rec.get("source_frames")
        or st.get("source_frame_count")
        or frames
        or 0
    )
    # Fallback: last progress total × stride ≈ source frames
    if src <= 0 and run.ticks:
        last = run.ticks[-1]
        stride = int(st.get("frame_stride") or pipe.get("frame_stride") or 1)
        if last.total > 0:
            src = int(last.total) * max(1, stride)
            if frames <= 0:
                frames = int(last.current or last.total) * max(1, stride)

    video_fps = float(rec.get("video_fps") or res.get("video_fps") or 0)
    video_sec = float(rec.get("video_duration_sec") or rec.get("duration_sec") or 0)
    if video_sec <= 0 and video_fps > 0 and src > 0:
        video_sec = src / video_fps

    elapsed = float(rec.get("elapsed_sec") or res.get("elapsed_sec") or 0)
    if elapsed <= 0 and progress.get("elapsed_sec"):
        elapsed = float(progress["elapsed_sec"])
    if elapsed <= 0 and run.ticks:
        elapsed = float(run.ticks[-1].elapsed_sec or 0)
    if elapsed <= 0:
        elapsed = float(run.wall_sec or 0)

    process_sec = elapsed  # server wall; client wall_sec kept separately
    fps_wall = float(rec.get("fps_processed") or res.get("fps_processed") or 0)
    if fps_wall <= 0 and process_sec > 0 and frames > 0:
        fps_wall = frames / process_sec

    elapsed_infer = float(st.get("elapsed_infer_sec") or pipe.get("elapsed_infer_sec") or 0)
    elapsed_pass2 = float(st.get("elapsed_pass2_sec") or pipe.get("elapsed_pass2_sec") or 0)
    elapsed_preload = float(st.get("elapsed_preload_sec") or pipe.get("elapsed_preload_sec") or 0)
    elapsed_finalize = float(st.get("elapsed_finalize_sec") or pipe.get("elapsed_finalize_sec") or 0)
    if elapsed_finalize <= 0 and process_sec > 0:
        elapsed_finalize = max(
            0.0,
            float(process_sec) - elapsed_preload - elapsed_infer - elapsed_pass2,
        )
    fps_infer = float(st.get("fps_infer") or 0)
    if fps_infer <= 0 and elapsed_infer > 0 and frames > 0:
        fps_infer = frames / elapsed_infer

    ratio = (process_sec / video_sec) if video_sec > 0 else None
    upload_sec = float(getattr(run, "upload_sec", 0.0) or 0.0)
    process_poll_sec = float(getattr(run, "process_poll_sec", 0.0) or 0.0)
    wall_sec = float(run.wall_sec or 0.0)
    upload_plus_process = upload_sec + process_sec
    ratio_wall = (wall_sec / video_sec) if video_sec > 0 else None
    ratio_up = (upload_plus_process / video_sec) if video_sec > 0 else None

    gs = rec.get("gpu_stats") or {}
    pm = st.get("process_memory") or gs.get("process_memory") or {}

    def _pm(key: str, default=None):
        if not isinstance(pm, dict):
            return default
        v = pm.get(key)
        return v if v is not None else default

    return {
        "label": run.label,
        "status": run.status,
        "job_id": run.job_id,
        "mode": "upload" if getattr(run, "use_upload", True) else "path",
        "frames": frames,
        "source_frames": src,
        "video_fps": round(video_fps, 3) if video_fps else None,
        "video_sec": round(video_sec, 3) if video_sec else None,
        "upload_sec": round(upload_sec, 3),
        "process_sec": round(process_sec, 3),
        "process_poll_sec": round(process_poll_sec, 3),
        "upload_plus_process_sec": round(upload_plus_process, 3),
        "wall_sec": round(wall_sec, 3),
        "ratio": round(ratio, 3) if ratio is not None else None,
        "ratio_process_video": round(ratio, 3) if ratio is not None else None,
        "ratio_wall_video": round(ratio_wall, 3) if ratio_wall is not None else None,
        "ratio_upload_process_video": round(ratio_up, 3) if ratio_up is not None else None,
        "frame_stride": int(st.get("frame_stride") or pipe.get("frame_stride") or 1),
        "elapsed_sec": round(elapsed, 3),
        "elapsed_infer_sec": round(elapsed_infer, 3),
        "elapsed_pass2_sec": round(elapsed_pass2, 3),
        "elapsed_preload_sec": round(elapsed_preload, 3),
        "elapsed_finalize_sec": round(elapsed_finalize, 3),
        "fps_processed": round(fps_wall, 2),
        "fps_infer": round(fps_infer, 2),
        "stage_decode_sec": round(float(st.get("stage_decode_sec") or pipe.get("stage_decode_sec") or 0), 3),
        "stage_gpu_infer_sec": round(float(st.get("stage_gpu_infer_sec") or pipe.get("stage_gpu_infer_sec") or 0), 3),
        "stage_cpu_finalize_sec": round(float(st.get("stage_cpu_finalize_sec") or pipe.get("stage_cpu_finalize_sec") or 0), 3),
        "use_sam_identity": pipe.get("use_sam_identity"),
        "use_reid": pipe.get("use_reid"),
        "use_offline_tracklet_link": pipe.get("use_offline_tracklet_link"),
        "models_reid": (rec.get("models") or {}).get("reid"),
        "process_rss_peak_mb": _pm("process_rss_peak_mb"),
        "process_rss_delta_peak_mb": _pm("process_rss_delta_peak_mb"),
        "cuda_allocated_peak_mb": _pm("cuda_allocated_peak_mb"),
        "cuda_reserved_peak_mb": _pm("cuda_reserved_peak_mb"),
        "gpu_device_peak_mb": gs.get("peak_gpu_device_used_mb") or gs.get("peak_mem_used_mb"),
        "avg_gpu_util_pct": gs.get("avg_gpu_util_pct"),

        "peak_gpu_util_pct": gs.get("peak_gpu_util_pct"),
        "error": run.error,
    }


class DockerApi:
    def __init__(self, base: str) -> None:
        self.base = base.rstrip("/")
        self.s = requests.Session()

    def health(self) -> dict:
        r = self.s.get(f"{self.base}/health", timeout=30)
        r.raise_for_status()
        return r.json()

    def upload_job(self, video: Path, *, prompt: str, max_sec: float | None) -> str:
        data: dict[str, str] = {"prompt": prompt}
        if max_sec is not None:
            data["max_duration_seconds"] = str(max_sec)
        with video.open("rb") as f:
            r = self.s.post(
                f"{self.base}/v1/jobs/upload",
                data=data,
                files={"file": (video.name, f, "application/octet-stream")},
                timeout=3600,
            )
        if r.status_code >= 400:
            raise ApiError(f"upload {r.status_code}: {r.text}")
        return r.json()["job_id"]

    def path_job(self, video: Path, *, prompt: str, max_sec: float | None) -> str:
        body: dict[str, Any] = {"path": str(video.resolve()), "prompt": prompt}
        if max_sec is not None:
            body["max_duration_seconds"] = max_sec
        r = self.s.post(f"{self.base}/v1/jobs", json=body, timeout=60)
        if r.status_code >= 400:
            raise ApiError(f"path job {r.status_code}: {r.text}")
        return r.json()["job_id"]

    def path_job_sync(self, video: Path, *, prompt: str, max_sec: float | None) -> dict:
        """One HTTP call; server waits on Event (no GET poll GIL storm)."""
        body: dict[str, Any] = {"path": str(video.resolve()), "prompt": prompt, "bench": True}
        if max_sec is not None:
            body["max_duration_seconds"] = max_sec
        r = self.s.post(f"{self.base}/v1/jobs/sync", json=body, timeout=7200)
        if r.status_code >= 400:
            raise ApiError(f"sync job {r.status_code}: {r.text}")
        return r.json()

    def job(self, job_id: str) -> dict:
        r = self.s.get(f"{self.base}/v1/jobs/{job_id}", timeout=30)
        r.raise_for_status()
        return r.json()

    def artifacts(self, job_id: str) -> list[dict]:
        r = self.s.get(f"{self.base}/v1/jobs/{job_id}/artifacts", timeout=60)
        r.raise_for_status()
        return r.json().get("files") or []

    def download(self, job_id: str, name: str) -> bytes:
        r = self.s.get(f"{self.base}/v1/jobs/{job_id}/artifacts/{name}", timeout=300)
        r.raise_for_status()
        return r.content

    def runs(self) -> list[dict]:
        r = self.s.get(f"{self.base}/v1/runs", timeout=30)
        r.raise_for_status()
        return r.json().get("runs") or []

    def poll_until_done(
        self,
        job_id: str,
        *,
        interval: float = 0.5,
        timeout: float = 7200,
        on_tick: Callable[[Tick], None] | None = None,
    ) -> tuple[dict, list[Tick]]:
        ticks: list[Tick] = []
        t0 = time.time()
        while time.time() - t0 < timeout:
            j = self.job(job_id)
            p = j.get("progress") or {}
            tick = Tick(
                t_wall=time.time() - t0,
                status=j.get("status", ""),
                current=int(p.get("current", 0)),
                total=int(p.get("total", 0)),
                percent=float(p.get("percent", 0)),
                fps=float(p.get("fps", 0)),
                elapsed_sec=float(p.get("elapsed_sec", 0)),
                eta_seconds=float(p.get("eta_seconds", 0)),
                gpu_mem_mb=float(p.get("cuda_allocated_mb", p.get("gpu_mem_mb", 0))),
                cuda_allocated_mb=float(p.get("cuda_allocated_mb", p.get("gpu_mem_mb", 0))),
                cuda_reserved_mb=float(p.get("cuda_reserved_mb", 0)),
                process_rss_mb=float(p.get("process_rss_mb", 0)),
                gpu_device_used_mb=float(p.get("gpu_device_used_mb", 0)),
                gpu_util_pct=float(p.get("gpu_util_pct", 0)),
                instances_peak=int(p.get("instances_peak", 0)),
                phase=str(p.get("phase") or "inference"),
            )
            ticks.append(tick)
            if on_tick:
                on_tick(tick)
            if tick.status in ("done", "error", "cancelled"):
                return j, ticks
            time.sleep(interval)
        raise ApiError(f"Job {job_id} timeout {timeout}s")



def _fmt_sec(v) -> str:
    if v is None:
        return "n/a"
    try:
        return f"{float(v):.3f}"
    except (TypeError, ValueError):
        return "n/a"


def _fmt_ratio(v) -> str:
    if v is None:
        return "n/a"
    try:
        return f"{float(v):.3f}×"
    except (TypeError, ValueError):
        return "n/a"


def _pct(part, whole) -> str:
    try:
        p = float(part)
        w = float(whole)
        if w <= 0:
            return "n/a"
        return f"{100.0 * p / w:.1f}%"
    except (TypeError, ValueError):
        return "n/a"


def format_benchmark_report(runs: list[RunResult], *, title: str = "YOLO_DRT API Benchmark Report") -> str:
    """Идеальный markdown-отчёт: каждая фаза отдельно (сек + % wall)."""
    rows = [_extract_speed(r) for r in runs]
    lines = [
        f"# {title}",
        "",
        f"- API: `{API_BASE}`",
        f"- Video: `{VIDEO_PATH}`",
        f"- Prompt: `{PROMPT}` | max_duration: `{MAX_DURATION_SECONDS}`",
        f"- USE_UPLOAD default: `{USE_UPLOAD}`",
        "",
        "## Фазы обработки (для каждого прогона)",
        "",
    ]
    for s in rows:
        wall = float(s.get("wall_sec") or 0)
        proc = float(s.get("process_sec") or 0)
        base = wall if wall > 0 else proc
        up = float(s.get("upload_sec") or 0)
        pre = float(s.get("elapsed_preload_sec") or 0)
        inf = float(s.get("elapsed_infer_sec") or 0)
        p2 = float(s.get("elapsed_pass2_sec") or 0)
        fin = float(s.get("elapsed_finalize_sec") or 0)
        vid = s.get("video_sec")
        lines += [
            f"### {s.get('label')} ({s.get('mode')})",
            "",
            "| Фаза | сек | % от wall | По-русски |",
            "|------|-----|-----------|-----------|",
            f"| upload/create | {_fmt_sec(up)} | {_pct(up, base)} | Загрузка файла / создание job |",
            f"| preload | {_fmt_sec(pre)} | {_pct(pre, base)} | Декод видео в оперативку |",
            f"| infer / YOLO Pass1 | {_fmt_sec(inf)} | {_pct(inf, base)} | Детекция + трекинг (только YOLO) |",
            f"| Pass2 tracklet/OSNet | {_fmt_sec(p2)} | {_pct(p2, base)} | Склейка треков после YOLO |",
            f"| finalize | {_fmt_sec(fin)} | {_pct(fin, base)} | Запись JSON / packets |",
            f"| process (server) | {_fmt_sec(proc)} | {_pct(proc, base)} | Всё на сервере |",
            f"| wall (client) | {_fmt_sec(wall)} | 100% | Полное время у клиента |",
            f"| video_sec | {_fmt_sec(vid)} | — | Длительность исходного ролика |",
            f"| ratio process/video | {_fmt_ratio(s.get('ratio_process_video'))} | — | process / длина ролика |",
            f"| ratio wall/video | {_fmt_ratio(s.get('ratio_wall_video'))} | — | wall / длина ролика |",
            "",
            "**Память (этот процесс API, не вся карта):**",
            "",
            "| Метрика | значение |",
            "|---------|----------|",
            f"| RAM peak (process) | {s.get('process_rss_peak_mb')} MB (+{s.get('process_rss_delta_peak_mb')} vs start) |",
            f"| CUDA alloc peak (torch) | {s.get('cuda_allocated_peak_mb')} MB |",
            f"| CUDA reserved peak | {s.get('cuda_reserved_peak_mb')} MB |",
            f"| GPU VRAM total (NVML, device) | {s.get('gpu_device_peak_mb')} MB |",
            "",
        ]

    lines += [
        "## Сводная таблица",
        "",
        "| label | mode | video | upload | preload | infer | pass2 | finalize | process | wall | rss_pk | cuda_pk | gpu_dev | p/v | w/v | fps_infer |",
        "|-------|------|-------|--------|---------|-------|-------|----------|---------|------|--------|---------|---------|-----|-----|-----------|",
    ]
    for s in rows:
        lines.append(
            "| {label} | {mode} | {video} | {up} | {pre} | {inf} | {p2} | {fin} | {proc} | {wall} | {rss} | {cuda} | {gdev} | {rp} | {rw} | {fi} |".format(
                label=s.get("label"),
                mode=s.get("mode"),
                video=_fmt_sec(s.get("video_sec")),
                up=_fmt_sec(s.get("upload_sec")),
                pre=_fmt_sec(s.get("elapsed_preload_sec")),
                inf=_fmt_sec(s.get("elapsed_infer_sec")),
                p2=_fmt_sec(s.get("elapsed_pass2_sec")),
                fin=_fmt_sec(s.get("elapsed_finalize_sec")),
                proc=_fmt_sec(s.get("process_sec")),
                wall=_fmt_sec(s.get("wall_sec")),
                rss=s.get("process_rss_peak_mb"),
                cuda=s.get("cuda_allocated_peak_mb"),
                gdev=s.get("gpu_device_peak_mb"),
                rp=_fmt_ratio(s.get("ratio_process_video")),
                rw=_fmt_ratio(s.get("ratio_wall_video")),
                fi=s.get("fps_infer"),
            )
        )
    lines += [
        "",
        "## Как читать (простыми словами)",
        "",
        "- **upload/create** — только отправка файла или создание задачи по пути",
        "- **preload** — чтение/декод ролика в RAM",
        "- **infer** — чистый YOLO Pass1 (детект+трек), без preload и без Pass2",
        "- **Pass2** — offline tracklet / OSNet в конце",
        "- **finalize** — запись JSON/packets после Pass2 (или остаток process)",
        "- **video_sec** — длина исходного видео",
        "- **ratio < 1×** — быстрее realtime; **> 1×** — медленнее длины ролика",
        "",
    ]
    if rows:
        main = next((r for r in rows if r.get("label") == "main"), rows[-1])
        lines.append("## Highlights")
        lines.append("")
        lines.append(
            f"- Main process/video = **{_fmt_ratio(main.get('ratio_process_video'))}** "
            f"(process `{_fmt_sec(main.get('process_sec'))}` / video `{_fmt_sec(main.get('video_sec'))}`)"
        )
        lines.append(
            f"- Main: upload `{_fmt_sec(main.get('upload_sec'))}` | "
            f"preload `{_fmt_sec(main.get('elapsed_preload_sec'))}` | "
            f"infer `{_fmt_sec(main.get('elapsed_infer_sec'))}` | "
            f"pass2 `{_fmt_sec(main.get('elapsed_pass2_sec'))}` | "
            f"finalize `{_fmt_sec(main.get('elapsed_finalize_sec'))}`"
        )
        lines.append("")
    return "\n".join(lines)


api = DockerApi(API_BASE)
print("API_BASE =", API_BASE)
print("VIDEO    =", VIDEO_PATH)
print("USE_UPLOAD =", USE_UPLOAD)

## 1. Health — готовность и флаги identity

In [ ]:
health = api.health()
paths = health.get("paths") or {}
print("status:", health.get("status"))
print("engines_ready:", health.get("engines_ready"))
print("--- PATHS (verify work_dir / upload_dir) ---")
for k in ("work_dir", "upload_dir", "output_dir", "detect_engine", "detect_engine_exists"):
    print(f"  {k}: {paths.get(k)}")
upload = str(paths.get("upload_dir") or "")
if "output" in upload.replace("\\", "/").lower() and "/uploads" in upload.replace("\\", "/").lower():
    print("WARN: upload_dir still under output/ — old bind path, restart API after pull")
if "\\\\work\\\\" in upload.replace("/", "\\") or "/work/" in upload.replace("\\", "/"):
    print("NOTE: upload_dir under project work/ — on host prefer TEMP or USE_UPLOAD=False for fair FPS")
print("--- identity / speed flags ---")
for k in (
    "use_sam_identity",
    "use_reid",
    "use_offline_tracklet_link",
    "tracklet_link_use_reid",
    "realtime_mode",
    "frame_source_mode",
):
    if k in paths:
        print(f"  {k}: {paths.get(k)}")
# Also dump nested processor / settings keys if present
proc = health.get("processor") or {}
print("processor.loaded:", proc.get("loaded"), "holders:", proc.get("holder_count"))
print("build_logs tail:")
for line in (health.get("build_logs") or [])[-6:]:
    print(" ", line)


## 2. Запуск job + сохранение результата

In [ ]:
def run_api_benchmark(
    *,
    label: str = "run",
    fetch_artifacts: bool = True,
    use_upload: bool | None = None,
) -> RunResult:
    do_upload = USE_UPLOAD if use_upload is None else use_upload
    mode = "upload" if do_upload else "path"
    print(f"=== {label} === mode={mode} video={VIDEO_PATH}")

    if do_upload and not VIDEO_PATH.is_file():
        raise FileNotFoundError(VIDEO_PATH)

    t0 = time.time()
    t_up0 = time.time()
    ticks: list[Tick] = []
    use_sync = (not do_upload) and bool(globals().get("USE_SYNC_BENCH", False))
    if do_upload:
        job_id = api.upload_job(
            VIDEO_PATH, prompt=PROMPT, max_sec=MAX_DURATION_SECONDS
        )
        upload_sec = time.time() - t_up0
        print(f"job_id: {job_id}  upload/create={upload_sec:.3f}s")
        _tick_last = {"key": None}

        def _tick(t: Tick) -> None:
            phase = getattr(t, "phase", "") or "inference"
            key = (t.status, phase, int(t.current), int(t.total))
            if key == _tick_last["key"]:
                return
            step = max(16, int(t.total) // 20) if t.total else 32
            show = (
                t.current == 0
                or t.current == t.total
                or phase in ("preload", "gpu", "start", "warmup", "inference", "staging")
            )
            if phase == "preload" and t.current not in (0, t.total) and t.current % step != 0:
                show = False
            if show:
                _tick_last["key"] = key
                print(
                    f"  [{t.status}/{phase}] {t.current}/{t.total} "
                    f"fps={t.fps:.1f} cuda={t.cuda_allocated_mb:.0f}MB rss={t.process_rss_mb:.0f}MB gpu%={t.gpu_util_pct:.0f} eta={t.eta_seconds:.0f}s"
                )

        t_poll0 = time.time()
        job, ticks = api.poll_until_done(
            job_id,
            interval=POLL_INTERVAL_SEC,
            on_tick=_tick,
        )
        process_poll_sec = time.time() - t_poll0
    elif use_sync:
        print("mode=path+SYNC (/v1/jobs/sync, no poll)")
        t_poll0 = time.time()
        job = api.path_job_sync(
            VIDEO_PATH, prompt=PROMPT, max_sec=MAX_DURATION_SECONDS
        )
        process_poll_sec = time.time() - t_poll0
        upload_sec = 0.0
        job_id = str(job.get("job_id") or "")
        print(f"job_id: {job_id}  sync_wait={process_poll_sec:.3f}s")
    else:
        job_id = api.path_job(
            VIDEO_PATH, prompt=PROMPT, max_sec=MAX_DURATION_SECONDS
        )
        upload_sec = time.time() - t_up0
        print(f"job_id: {job_id}  upload/create={upload_sec:.3f}s  (polling)")
        _tick_last = {"key": None}

        def _tick(t: Tick) -> None:
            phase = getattr(t, "phase", "") or "inference"
            key = (t.status, phase, int(t.current), int(t.total))
            if key == _tick_last["key"]:
                return
            step = max(16, int(t.total) // 20) if t.total else 32
            show = (
                t.current == 0
                or t.current == t.total
                or phase in ("preload", "gpu", "start", "warmup", "inference", "staging")
            )
            if phase == "preload" and t.current not in (0, t.total) and t.current % step != 0:
                show = False
            if show:
                _tick_last["key"] = key
                print(
                    f"  [{t.status}/{phase}] {t.current}/{t.total} "
                    f"fps={t.fps:.1f} cuda={t.cuda_allocated_mb:.0f}MB rss={t.process_rss_mb:.0f}MB gpu%={t.gpu_util_pct:.0f} eta={t.eta_seconds:.0f}s"
                )

        t_poll0 = time.time()
        job, ticks = api.poll_until_done(
            job_id,
            interval=POLL_INTERVAL_SEC,
            on_tick=_tick,
        )
        process_poll_sec = time.time() - t_poll0
    wall = time.time() - t0
    err = None
    if job.get("status") == "error":
        err = (job.get("result") or {}).get("error") or job.get("error")
        print("ERROR:", err)

    files: dict[str, bytes] = {}
    if fetch_artifacts and job.get("status") == "done":
        for item in api.artifacts(job_id):
            name = item.get("name") or item.get("filename")
            if not name:
                continue
            try:
                files[name] = api.download(job_id, name)
            except Exception as exc:
                print(f"artifact skip {name}: {exc}")

    run = RunResult(
        job_id=job_id,
        wall_sec=wall,
        status=job.get("status", ""),
        job=job,
        ticks=ticks,
        files=files,
        error=err,
        label=label,
        upload_sec=upload_sec,
        process_poll_sec=process_poll_sec,
        use_upload=do_upload,
    )
    speed = _extract_speed(run)
    print(
        f"DONE status={run.status} | "
        f"video={_fmt_sec(speed.get('video_sec'))}s "
        f"upload={_fmt_sec(speed.get('upload_sec'))}s "
        f"preload={_fmt_sec(speed.get('elapsed_preload_sec'))}s "
        f"infer={_fmt_sec(speed.get('elapsed_infer_sec'))}s "
        f"pass2={_fmt_sec(speed.get('elapsed_pass2_sec'))}s "
        f"finalize={_fmt_sec(speed.get('elapsed_finalize_sec'))}s "
        f"process={_fmt_sec(speed.get('process_sec'))}s "
        f"wall={_fmt_sec(speed.get('wall_sec'))}s | "
        f"ratio_p/v={_fmt_ratio(speed.get('ratio_process_video'))} "
        f"ratio_w/v={_fmt_ratio(speed.get('ratio_wall_video'))} | "
        f"fps_infer={speed['fps_infer']} fps_wall={speed['fps_processed']} | "
        f"stages decode={_fmt_sec(speed.get('stage_decode_sec'))}s "
        f"gpu_wall={_fmt_sec(speed.get('stage_gpu_infer_sec'))}s "
        f"cpu_fin={_fmt_sec(speed.get('stage_cpu_finalize_sec'))}s | "
        f"rss_pk={speed.get('process_rss_peak_mb')}MB "
        f"cuda_pk={speed.get('cuda_allocated_peak_mb')}MB "
        f"gpu_dev={speed.get('gpu_device_peak_mb')}MB"
    )

    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    out = RESULTS_DIR / f"{stamp}_{label}.json"
    out.write_text(
        json.dumps(
            {
                "label": label,
                "api_base": API_BASE,
                "use_upload": do_upload,
                "video": str(VIDEO_PATH),
                "speed": speed,
                "job": job,
                "ticks": [t.__dict__ for t in ticks],
            },
            ensure_ascii=False,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )
    print("saved:", out.resolve())
    return run

In [ ]:
ALL: list[RunResult] = []
ALL_INCLUDING_WARMUP: list[RunResult] = []

if WARMUP_RUN:
    print("--- warmup (excluded from chart summary; included in REPORT) ---")
    warm = run_api_benchmark(label="warmup", fetch_artifacts=False)
    ALL_INCLUDING_WARMUP.append(warm)

RUN = run_api_benchmark(label="main", fetch_artifacts=True)
ALL.append(RUN)
ALL_INCLUDING_WARMUP.append(RUN)

for i in range(int(REPEAT_RUNS)):
    r = run_api_benchmark(label=f"repeat_{i+1}", fetch_artifacts=False)
    ALL.append(r)
    ALL_INCLUDING_WARMUP.append(r)

summary_df = pd.DataFrame([_extract_speed(r) for r in ALL])
compare_df = pd.DataFrame([_extract_speed(r) for r in ALL_INCLUDING_WARMUP])

display(Markdown("### Сводка прогонов (без warmup в графиках)"))
cols = [
    "label",
    "mode",
    "video_sec",
    "upload_sec",
    "process_sec",
    "wall_sec",
    "ratio_process_video",
    "ratio_wall_video",
    "ratio_upload_process_video",
    "fps_infer",
    "fps_processed",
    "elapsed_preload_sec",
    "elapsed_infer_sec",
    "elapsed_pass2_sec",
    "elapsed_finalize_sec",
    "avg_gpu_util_pct",
    "process_rss_peak_mb",
    "cuda_allocated_peak_mb",
    "gpu_device_peak_mb",
]
display(summary_df[[c for c in cols if c in summary_df.columns]])

display(Markdown("### Comparison table (warmup + main + repeats)"))
display(compare_df[[c for c in cols if c in compare_df.columns]])

REPORT_MD = format_benchmark_report(ALL_INCLUDING_WARMUP)
display(Markdown("### REPORT (copy-paste)"))
display(Markdown(REPORT_MD))

def _write_reports(md: str) -> None:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    report_path = RESULTS_DIR / f"report_{stamp}.md"
    latest_path = RESULTS_DIR / "report_latest.md"
    report_path.write_text(md, encoding="utf-8")
    latest_path.write_text(md, encoding="utf-8")
    csv_path = RESULTS_DIR / "last_summary.csv"
    compare_df.to_csv(csv_path, index=False)
    print("REPORT saved:", report_path.resolve())
    print("REPORT latest:", latest_path.resolve())
    print("CSV saved:", csv_path.resolve())
    if not report_path.is_file() or report_path.stat().st_size < 50:
        raise RuntimeError(f"Report write failed or empty: {report_path}")

try:
    _write_reports(REPORT_MD)
except Exception as exc:
    print("FATAL: report write failed:", exc)
    raise

display(
    Markdown(
        "**Как читать:** upload → preload → infer(YOLO) → Pass2 → finalize. "
        "`ratio_process_video` &lt;1× = быстрее realtime. Полный разбор — `report_latest.md`."
    )
)

## 3. Progress во время job (polling)

In [ ]:
df = pd.DataFrame([t.__dict__ for t in RUN.ticks])
if df.empty:
    display(Markdown(
        "**Sync mode:** polling ticks не собирались. Память — §5 (`process_memory_samples`) и `stats_summary.process_memory`."
    ))
else:
    n_ax = 4 if "process_rss_mb" in df.columns else 3
    fig, axes = plt.subplots(n_ax, 1, figsize=(12, 3 * n_ax), sharex=True)
    if n_ax == 1:
        axes = [axes]
    fig.suptitle(f"Job {RUN.job_id[:8]}…  status={RUN.status}")

    axes[0].plot(df["t_wall"], df["fps"], label="progress fps")
    axes[0].set_ylabel("FPS")
    axes[0].legend(loc="upper right")
    axes[0].grid(True, alpha=0.3)

    if n_ax >= 4:
        axes[1].plot(df["t_wall"], df["process_rss_mb"], label="process RSS MB")
        axes[1].plot(df["t_wall"], df["cuda_allocated_mb"], label="CUDA alloc MB")
        if "gpu_device_used_mb" in df.columns:
            axes[1].plot(df["t_wall"], df["gpu_device_used_mb"], label="device VRAM (NVML)", alpha=0.5)
        axes[1].set_ylabel("MB")
        axes[1].legend(loc="upper left", fontsize=8)
        axes[1].grid(True, alpha=0.3)
        ax_gpu = axes[2]
        ax_pct = axes[3]
    else:
        ax_gpu = axes[1]
        ax_pct = axes[2]

    ax_gpu.plot(df["t_wall"], df["gpu_util_pct"], color="C1", label="gpu util %")
    ax_gpu.set_ylabel("GPU %")
    ax_gpu.legend(loc="upper right")
    ax_gpu.grid(True, alpha=0.3)

    ax_pct.plot(df["t_wall"], df["percent"], color="C2", label="percent")
    ax_pct.set_ylabel("%")
    ax_pct.set_xlabel("wall sec (client poll)")
    ax_pct.legend(loc="upper right")
    ax_pct.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    med = f"progress fps median={df['fps'].median():.1f}  gpu util median={df['gpu_util_pct'].median():.1f}%"
    if "cuda_allocated_mb" in df.columns:
        med += f"  cuda peak poll={df['cuda_allocated_mb'].max():.0f}MB  rss peak poll={df['process_rss_mb'].max():.0f}MB"
    print(med)

## 4. Идеальный разбор фаз

Таблица: upload | preload | infer | pass2 | finalize | video | ratios.

Stacked bar: **upload | preload | infer | pass2 | finalize**.


In [ ]:
speed = _extract_speed(RUN)
rec = RUN.record
pipe = RUN.pipeline
gs = rec.get("gpu_stats") or {}
models = rec.get("models") or {}

wall = float(speed.get("wall_sec") or 0) or float(speed.get("process_sec") or 0) or 1.0
phases = [
    ("upload/create", float(speed.get("upload_sec") or 0), "Загрузка / create job"),
    ("preload", float(speed.get("elapsed_preload_sec") or 0), "Декод в RAM"),
    ("infer / YOLO Pass1", float(speed.get("elapsed_infer_sec") or 0), "Детект + трек"),
    ("Pass2 tracklet/OSNet", float(speed.get("elapsed_pass2_sec") or 0), "Склейка треков"),
    ("finalize", float(speed.get("elapsed_finalize_sec") or 0), "JSON / packets"),
]

md_lines = [
    "### REPORT — фазы main-прогона",
    "",
    "| Фаза | сек | % wall | По-русски |",
    "|------|-----|--------|-----------|",
]
for name, sec, ru in phases:
    md_lines.append(f"| {name} | {_fmt_sec(sec)} | {_pct(sec, wall)} | {ru} |")
md_lines += [
    f"| **video_sec** | {_fmt_sec(speed.get('video_sec'))} | — | Длина ролика |",
    f"| **process_sec** | {_fmt_sec(speed.get('process_sec'))} | {_pct(speed.get('process_sec'), wall)} | Сервер целиком |",
    f"| **wall_sec** | {_fmt_sec(speed.get('wall_sec'))} | 100% | Клиент end-to-end |",
    f"| **ratio process/video** | {_fmt_ratio(speed.get('ratio_process_video'))} | — | process / video |",
    f"| **ratio wall/video** | {_fmt_ratio(speed.get('ratio_wall_video'))} | — | wall / video |",
]
display(Markdown('\n'.join(md_lines)))

rows = [
    ("video_sec", speed.get("video_sec")),
    ("upload_sec", speed.get("upload_sec")),
    ("elapsed_preload_sec", speed.get("elapsed_preload_sec")),
    ("elapsed_infer_sec (YOLO only)", speed.get("elapsed_infer_sec")),
    ("elapsed_pass2_sec", speed.get("elapsed_pass2_sec")),
    ("elapsed_finalize_sec", speed.get("elapsed_finalize_sec")),
    ("process_sec (server)", speed.get("process_sec")),
    ("wall_sec (client)", speed.get("wall_sec")),
    ("ratio process/video", _fmt_ratio(speed.get("ratio_process_video"))),
    ("ratio wall/video", _fmt_ratio(speed.get("ratio_wall_video"))),
    ("fps_infer (YOLO only)", speed["fps_infer"]),
    ("fps_processed (wall)", speed["fps_processed"]),
    ("frames / source", f"{speed['frames']} / {speed['source_frames']}"),
    ("frame_stride", speed["frame_stride"]),
    ("use_sam_identity", speed["use_sam_identity"]),
    ("use_reid (flag)", speed["use_reid"]),
    ("models.reid", models.get("reid")),
    ("offline_tracklet_link", speed["use_offline_tracklet_link"]),
    ("avg_gpu_util_pct", gs.get("avg_gpu_util_pct")),
    ("peak_gpu_util_pct", gs.get("peak_gpu_util_pct")),
    ("process_rss_peak_mb", speed.get("process_rss_peak_mb")),
    ("cuda_allocated_peak_mb", speed.get("cuda_allocated_peak_mb")),
    ("gpu_device_peak_mb", speed.get("gpu_device_peak_mb")),
    ("resolution", rec.get("resolution")),
]
display(Markdown("### Полный разбор"))
display(pd.DataFrame(rows, columns=["metric", "value"]))

up_s, pre_s, inf_s, p2_s, fin_s = [p[1] for p in phases]
if up_s + pre_s + inf_s + p2_s + fin_s > 0:
    fig, ax = plt.subplots(figsize=(10, 2.8))
    left = 0.0
    colors = ["#4c78a8", "#f58518", "#54a24b", "#e45756", "#b279a2"]
    labels = [
        f"upload {up_s:.2f}s",
        f"preload {pre_s:.2f}s",
        f"infer {inf_s:.2f}s",
        f"pass2 {p2_s:.2f}s",
        f"finalize {fin_s:.2f}s",
    ]
    for val, lab, col in zip([up_s, pre_s, inf_s, p2_s, fin_s], labels, colors):
        ax.barh(["фазы"], [val], left=left, label=lab, color=col)
        left += val
    ax.set_xlabel("секунды")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.35), ncol=5, frameon=False)
    ax.set_title("Stacked: upload | preload | infer | pass2 | finalize")
    plt.tight_layout()
    plt.show()
else:
    print("Нет фазовых таймингов в stats — перезапусти API с новым video_processor.")

fig2, ax2 = plt.subplots(figsize=(8, 3))
labels_b = ["video", "upload", "process", "wall"]
vals_b = [
    float(speed.get("video_sec") or 0),
    float(speed.get("upload_sec") or 0),
    float(speed.get("process_sec") or 0),
    float(speed.get("wall_sec") or 0),
]
ax2.bar(labels_b, vals_b, color=["#4c78a8", "#f58518", "#54a24b", "#e45756"])
ax2.set_ylabel("seconds")
ax2.set_title("Main: video vs upload vs process vs wall")
plt.tight_layout()
plt.show()

print("models:", models)

## 5. Память процесса + NVML (артефакты run)


In [ ]:
gpu_nvml = next((n for n in RUN.files if n.endswith("_gpu_samples.json")), None)
proc_name = next((n for n in RUN.files if n.endswith("_process_memory_samples.json")), None)

pm = (RUN.stats or {}).get("process_memory") or (RUN.record.get("gpu_stats") or {}).get("process_memory") or {}
if pm:
    display(Markdown(
        f"**Peaks (this process):** RAM {pm.get('process_rss_peak_mb')} MB "
        f"(+{pm.get('process_rss_delta_peak_mb')} MB), "
        f"CUDA alloc {pm.get('cuda_allocated_peak_mb')} MB, "
        f"reserved {pm.get('cuda_reserved_peak_mb')} MB"
    ))

if proc_name:
    pdf = pd.DataFrame(json.loads(RUN.files[proc_name].decode("utf-8")))
    print("process_memory_samples:", proc_name)
    display(pdf.head())
    if not pdf.empty and "t_sec" in pdf.columns:
        fig, ax = plt.subplots(figsize=(12, 3))
        ax.plot(pdf["t_sec"], pdf["process_rss_mb"], label="process RSS MB")
        ax.plot(pdf["t_sec"], pdf["cuda_allocated_mb"], label="CUDA alloc MB (this process)")
        ax.plot(pdf["t_sec"], pdf["cuda_reserved_mb"], label="CUDA reserved MB", alpha=0.7)
        ax.set_xlabel("t_sec")
        ax.set_ylabel("MB")
        ax.legend(loc="upper left")
        ax.set_title("Process memory (not whole-GPU NVML)")
        plt.tight_layout()
        plt.show()
else:
    print("Нет process_memory_samples — обнови API и перезапусти job")

if gpu_nvml:
    gdf = pd.DataFrame(json.loads(RUN.files[gpu_nvml].decode("utf-8")))
    print("gpu_samples (NVML device total):", gpu_nvml)
    display(gdf.head())
    if not gdf.empty and "t_sec" in gdf.columns:
        fig, ax = plt.subplots(figsize=(12, 3))
        ax.plot(gdf["t_sec"], gdf["gpu_util_pct"], label="GPU util %")
        ax.set_ylabel("util %")
        if "mem_used_mb" in gdf.columns:
            ax2 = ax.twinx()
            ax2.plot(gdf["t_sec"], gdf["mem_used_mb"], color="C3", alpha=0.5, label="device VRAM total")
            ax2.set_ylabel("device MB (all processes)")
        ax.set_xlabel("t_sec")
        ax.set_title("NVML — whole GPU (reference only)")
        plt.tight_layout()
        plt.show()
else:
    print("Нет gpu_samples (NVML off или артеfact не скачан)")

## 6. История API `/v1/runs`

In [ ]:
hist = pd.DataFrame(api.runs())
print(f"Runs in index: {len(hist)}")
if not hist.empty:
    cols = [
        c
        for c in [
            "run_id",
            "elapsed_sec",
            "fps_processed",
            "frames",
            "source_frames",
            "resolution",
        ]
        if c in hist.columns
    ]
    display(hist[cols].head(15))

summary_df.to_csv(RESULTS_DIR / "last_summary.csv", index=False)
print("CSV:", (RESULTS_DIR / "last_summary.csv").resolve())

## Честный compare с UI / Docker I/O

1. В UI возьми `fps` / время с того же ролика (baseline ~41 FPS на `video.mp4`).
2. Здесь смотри **`fps_infer`**, **`process_sec`**, **`upload_sec`**, ratios.
3. Хост fair: `API_BASE=http://127.0.0.1:8765` + **`USE_UPLOAD=False`** + тот же `VIDEO_PATH`.
4. Docker: `API_BASE=...:8080` + `USE_UPLOAD=True` (named volume) **или** файл в `YOLO_DOCKER/videos/` + `/data/videos/...`.
5. Перед прогоном: ячейка Health — проверь `paths.work_dir` / `paths.upload_dir`.
6. После прогона REPORT всегда в `notebooks/benchmark_results/report_latest.md` (+ `report_*.md`).
7. Если `fps_infer` ~18 при UI ~41 — смотри путь decode (upload на другой диск), не Pass2 (~3s).
